<a href="https://colab.research.google.com/github/abhijadhav14/Data-Analytics-Using-Python/blob/main/Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from sklearn.datasets import load_iris

In [2]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("IrisClassification") \
    .getOrCreate()

In [9]:
import pandas as pd

# Load the Iris dataset
iris = load_iris()

# Create a Pandas DataFrame from the Iris dataset's data and feature names
iris_df_pd = pd.DataFrame(data=iris.data, columns=iris.feature_names)

# Add the target (species) column to the Pandas DataFrame
# Map numerical targets to species names for readability
iris_df_pd['species'] = iris.target_names[iris.target]

# Convert the Pandas DataFrame to a Spark DataFrame
df = spark.createDataFrame(iris_df_pd)

print("Schema:")
df.printSchema()

print("\nFirst 5 rows:")
df.show(5)

print("\nColumns:")
print(df.columns)

Schema:
root
 |-- sepal length (cm): double (nullable = true)
 |-- sepal width (cm): double (nullable = true)
 |-- petal length (cm): double (nullable = true)
 |-- petal width (cm): double (nullable = true)
 |-- species: string (nullable = true)


First 5 rows:
+-----------------+----------------+-----------------+----------------+-------+
|sepal length (cm)|sepal width (cm)|petal length (cm)|petal width (cm)|species|
+-----------------+----------------+-----------------+----------------+-------+
|              5.1|             3.5|              1.4|             0.2| setosa|
|              4.9|             3.0|              1.4|             0.2| setosa|
|              4.7|             3.2|              1.3|             0.2| setosa|
|              4.6|             3.1|              1.5|             0.2| setosa|
|              5.0|             3.6|              1.4|             0.2| setosa|
+-----------------+----------------+-----------------+----------------+-------+
only showing top 5

In [10]:
# Convert class labels to numeric values
indexer = StringIndexer(
    inputCol="Species", # Changed from "class" to "species"
    outputCol="label"
)

df = indexer.fit(df).transform(df)

In [13]:
# Combine features
assembler = VectorAssembler(
    inputCols=[
        "sepal length (cm)",
        "sepal width (cm)",
        "petal length (cm)",
        "petal width (cm)"
    ],
    outputCol="features"
)

data = assembler.transform(df)

In [14]:
# Split dataset
train_data, test_data = data.randomSplit(
    [0.8, 0.2],
    seed=42
)

In [16]:
# Decision Tree Model
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label"
)

model = dt.fit(train_data)

In [17]:
# Predictions
predictions = model.transform(test_data)

predictions.select(
    "Species", # Changed from "class" to "species"
    "label",
    "prediction"
).show()

+----------+-----+----------+
|   Species|label|prediction|
+----------+-----+----------+
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|    setosa|  0.0|       0.0|
|versicolor|  1.0|       1.0|
|    setosa|  0.0|       0.0|
|versicolor|  1.0|       1.0|
|versicolor|  1.0|       1.0|
| virginica|  2.0|       1.0|
|versicolor|  1.0|       1.0|
|versicolor|  1.0|       1.0|
|versicolor|  1.0|       1.0|
+----------+-----+----------+
only showing top 20 rows


In [19]:
# Accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("Accuracy =", accuracy)
print("Accuracy Percentage =", round(accuracy * 100, 2), "%")

Accuracy = 0.96875
Accuracy Percentage = 96.88 %


In [20]:
spark.stop()